# Total Taxes per Person
State taxes plus city, county and K-12 school district taxes, per Californian. This counts taxes California governments *collect*, not what households pay (federal taxes are excluded, and some state tax is paid by businesses and visitors).

**Left out:** special districts and community colleges. School districts here means local property and other local taxes only; the state aid they receive (about half of school revenue) is already counted in state spending.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
SCO_NOTE = "State = all funds (LAO / DOF). County and city = SCO. School districts = Census (property + other local taxes). Excludes special districts and community colleges."
st = pd.read_csv(PROCESSED / "state_revenue.csv")
lt = pd.read_csv(PROCESSED / "local_taxes.csv")
sf = pd.read_csv(PROCESSED / "school_finance.csv")
a = st[st.group == "Tax"].groupby("fy").amount_k.sum().rename("State")
b = lt[lt.govt == "City"].groupby("fy").amount_k.sum().rename("City")
c = lt[lt.govt == "County"].groupby("fy").amount_k.sum().rename("County")
s_ = sf[(sf.kind == "revenue") & sf.group.isin(["Local property tax", "Other local taxes"])].groupby("fy").amount_k.sum().rename("School districts")
d = pd.concat([a, b, c, s_], axis=1).dropna().reset_index().melt("fy", var_name="level", value_name="amount_k")
d = add_views(d)
d = d[d.fy <= 2024]
stacked_area_toggle(d, "level", "Taxes collected: state, county, city and school districts", SCO_NOTE, order=["State", "County", "City", "School districts"]).show()

## As a share of personal income
Taxes collected divided by total personal income of California residents (BEA, via FRED; fiscal-year average of quarterly rates). This is a rough yardstick of how big government's take is relative to the economy, **not** an average household tax rate: businesses, visitors and non-residents also pay California taxes, and income here excludes corporate profits retained by firms.

In [2]:
mac = pd.read_csv(PROCESSED / "macro.csv").set_index("fy")
inc_k = mac.income_m * 1000  # $ millions -> $ thousands, same unit as amount_k
state = (a / inc_k).dropna() * 100
state = state[state.index <= LAST_ACTUAL_FY]
fig = go.Figure(go.Scatter(x=state.index, y=state, mode="lines", line=dict(color=PALETTE[0]), hovertemplate="%{y:.1f}%"))
fig.update_layout(yaxis_title="% of personal income", yaxis_ticksuffix="%", xaxis_title="Fiscal year (ending June 30)", showlegend=False)
base_layout(fig, "State taxes as a share of California personal income", "State taxes, all funds (LAO / DOF) over BEA personal income. Source: LAO, DOF, BEA via FRED")
fig.show()

In [3]:
parts = pd.concat([a, c, b, s_], axis=1, keys=["State", "County", "City", "School districts"]).dropna()
parts = parts[parts.index <= 2024]
pct = parts.div(inc_k.reindex(parts.index), axis=0) * 100
fig = go.Figure()
for i, col in enumerate(pct.columns):
    fig.add_trace(go.Scatter(x=pct.index, y=pct[col], name=col, stackgroup="one", mode="lines", line=dict(width=0.5, color=PALETTE[i]),
                             hovertemplate="%{y:.2f}%"))
fig.update_layout(yaxis_title="% of personal income", yaxis_ticksuffix="%", xaxis_title="Fiscal year (ending June 30)")
base_layout(fig, "All California government taxes as a share of personal income", "State + county + city + school districts. " + SCO_NOTE)
fig.show()